# MOONPIERCER — Methodology Figures

Data-independent illustrations of the chord-search pipeline methodology. These figures can be generated without HPC results.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from moonpiercer.config import ChordConfig
from moonpiercer.constants import LUNAR_RADIUS_M
from moonpiercer.geometry import chord_length_from_separation, expected_ellipticity_from_separation
from moonpiercer.velocity import maxwell_boltzmann_speed_pdf, rotation_offset_deg
from moonpiercer.plotting import plot_transit_cone_diagram, plot_chord_space_diagram
from moonpiercer.io_utils import save_figure, PAPER_FIGURES_DIR, ensure_dir
from moonpiercer.constants import (
    KAPPA_CLASSICAL_LOW_M2_PER_MYR,
    KAPPA_CLASSICAL_HIGH_M2_PER_MYR,
    KAPPA_ADOPTED_M2_PER_MYR,
)

%matplotlib inline

# Publication rcParams (ApJ / MNRAS 2-column format)
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 8,
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.minor.size": 1.5,
    "ytick.minor.size": 1.5,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.top": True,
    "ytick.right": True,
    "legend.fontsize": 7,
    "legend.framealpha": 0.9,
    "figure.constrained_layout.use": True,
})

# Paper figure output
ensure_dir(PAPER_FIGURES_DIR)

# Single-column width for 2-column journal
COL_W = 3.54  # inches (90 mm)
FULL_W = 7.25  # inches (184 mm)

cfg = ChordConfig()
print("Methodology notebook ready.")

## Transit Cone Diagram

Schematic showing the PBH transit geometry: entry point, exit point, and the cone of possible chord orientations through the Moon.

In [ ]:
fig = plot_transit_cone_diagram(
    entry_lon=30.0, entry_lat=15.0,
    ellipticity=1.8, orientation_deg=45.0,
    cone_half_deg=2.0,
    figsize=(FULL_W, FULL_W * 0.5),
)
save_figure(fig, "methodology_transit_cone", pdf_dir=PAPER_FIGURES_DIR)
plt.show()

## Chord Search-Space Diagram

Illustration of the search annulus around a candidate entry crater, showing the allowed region for a paired exit crater given the chord-length and orientation constraints.

In [ ]:
fig = plot_chord_space_diagram(
    config=cfg,
    entry_lon_deg=30.0, entry_lat_deg=15.0,
    orientation_deg=45.0, ellipticity=1.5,
    figsize=(COL_W, COL_W),
)
save_figure(fig, "methodology_chord_space", pdf_dir=PAPER_FIGURES_DIR)
plt.show()

## PBH Velocity Distribution

Maxwell-Boltzmann speed distribution for dark-matter particles in the Galactic halo, truncated at the local escape velocity.

In [ ]:
fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
v = np.linspace(0.1, 650, 2000)
pdf = maxwell_boltzmann_speed_pdf(v)
ax.plot(v, pdf, "k-", lw=1.0)
ax.fill_between(v, pdf, alpha=0.15, color="steelblue")
ax.axvline(220, color="C3", ls="--", lw=0.8, label=r"$v_0 = 220$ km s$^{-1}$")
ax.axvline(544, color="C0", ls="--", lw=0.8, label=r"$v_{\rm esc} = 544$ km s$^{-1}$")
ax.set_xlabel(r"PBH speed [km s$^{-1}$]")
ax.set_ylabel("Probability density")
ax.set_xlim(0, 650)
ax.set_ylim(bottom=0)
ax.legend()
save_figure(fig, "methodology_velocity_distribution", pdf_dir=PAPER_FIGURES_DIR)
plt.show()

## Rotation Offset vs Chord Length

The angular rotation of the Moon during a PBH transit as a function of chord length, for several representative PBH speeds.

In [ ]:
fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
seps = np.linspace(30, 180, 200)
L_km = chord_length_from_separation(seps) / 1e3
speeds = [(50, "C3", r"50 km s$^{-1}$"), (100, "C1", r"100 km s$^{-1}$"),
          (220, "C2", r"220 km s$^{-1}$"), (544, "C0", r"544 km s$^{-1}$")]
for v_km_s, color, label in speeds:
    offset = rotation_offset_deg(chord_length_from_separation(seps), v_km_s)
    ax.plot(L_km, offset * 3600, color=color, lw=1.0, label=label)

ax.set_xlabel("Chord length [km]")
ax.set_ylabel("Rotation offset [arcsec]")
ax.set_xlim(0, 3500)
ax.legend()
save_figure(fig, "methodology_rotation_offset", pdf_dir=PAPER_FIGURES_DIR)
plt.show()

## Expected Ellipticity vs Angular Separation

Geometric prediction for the ellipticity of impact craters as a function of the angular separation between entry and exit points on the lunar surface.

In [ ]:
fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
seps = np.linspace(20, 180, 200)
e = expected_ellipticity_from_separation(seps)
ax.plot(seps, e, "k-", lw=1.0)
ax.axhline(1.0, color="0.6", ls=":", lw=0.5)
ax.set_xlabel(r"Angular separation [deg]")
ax.set_ylabel("Expected crater ellipticity")
ax.set_ylim(0.9, 6)
ax.set_xlim(20, 180)
save_figure(fig, "methodology_ellipticity_vs_separation", pdf_dir=PAPER_FIGURES_DIR)
plt.show()

## Crater Survival Timescale

Estimated survival timescale of small lunar craters under diffusive degradation, for three representative values of the regolith diffusivity $\kappa$.

In [ ]:
fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
r_range = np.linspace(0.5, 15, 200)

kappas = [
    (KAPPA_CLASSICAL_LOW_M2_PER_MYR, r"$\kappa = 0.5$ m$^2$ Myr$^{-1}$ (low)", "--", "C0"),
    (KAPPA_ADOPTED_M2_PER_MYR, r"$\kappa = 1.6$ m$^2$ Myr$^{-1}$ (adopted)", "-", "k"),
    (KAPPA_CLASSICAL_HIGH_M2_PER_MYR, r"$\kappa = 5.5$ m$^2$ Myr$^{-1}$ (high)", ":", "C3"),
]
for kap, label, ls, color in kappas:
    tau = r_range ** 2 / (4 * kap)
    ax.plot(r_range, tau, ls=ls, color=color, lw=1.0, label=label)

ax.set_xlabel("Crater radius [m]")
ax.set_ylabel(r"$\tau_{\rm surv}$ [Myr]")
ax.set_ylim(0, 15)
ax.set_xlim(0.5, 15)
ax.legend()

# Mark typical detection range
ax.axvspan(1.0, 10.0, alpha=0.06, color="gold", label="Detection range")

save_figure(fig, "methodology_survival_timescale", pdf_dir=PAPER_FIGURES_DIR)
plt.show()